# Welcome — Using the Memories.AI API

This Jupyter notebook guides you through using the [Memories.AI API](https:memories.ai/docs). It contains upload, search, videochat, transcribe, and several utility tools, for you to run and adapt.

## Prerequisites

- Python 3.8+ running in a Jupyter environment (Notebook, JupyterLab, or Colab).
- Package:
  - `requests`
  - `json`

- An API key: Save it here.


In [ ]:
API_KEY = "" # Your memories.ai API key

This is an example video (and its URL) that we will be working on. You can click run and see the video.


In [ ]:
from IPython.display import HTML
# Note that you don't need to modify this block.
VIDEO_URL = "https://commondatastorage.googleapis.com/gtv-videos-bucket/sample/ElephantsDream.mp4"

html = f"""
<div style="max-width:720px;">
  <video controls playsinline style="width:100%; height:auto;" preload="metadata" poster="">
    <source src="{VIDEO_URL}" type="video/mp4">
    Your browser does not support HTML5 video. <a href="{VIDEO_URL}">Download</a>
  </video>
</div>
"""
display(HTML(html))

## Upload

Let’s start by uploading the video from the URL shown above.

If the upload is successful, the returned JSON will include:

```json
"videoNo": "...",
"videoName": "...",
"success": true


In [2]:
import requests  
headers = {"Authorization": API_KEY} 

# Video file details  
data = {  
    "url": "http://commondatastorage.googleapis.com/gtv-videos-bucket/sample/ElephantsDream.mp4"
    # "callback": "<YOUR_CALLBACK_URI>" # Optional
}  

response = requests.post(  
    "https://api.memories.ai/serve/api/v1/upload_url",  
    data=data,  
    headers=headers  
)  

print(response.json())  

{'code': '0000', 'msg': 'success', 'data': {'videoNo': 'VI627384556951646208', 'videoName': 'ElephantsDream'}, 'success': True, 'failed': False}


## Check video parsing status

Before proceeding, confirm that the video has been fully parsed by our model.  

Run the next code cell to check the status.  
Continue only when you see:

```json
"status": "PARSE"

In [9]:
headers = {"Authorization": API_KEY} 
params = {
    "video_no": "VI627384556951646208"  # Replace with the actual video number from the upload response,
}
response = requests.post("https://api.memories.ai/serve/api/v1/list_videos", headers=headers, json=params)
print(response.json())

{'code': '0000', 'msg': 'success', 'data': {'videos': [{'duration': '653', 'size': '169612362', 'status': 'PARSE', 'cause': 'null', 'video_no': 'VI627384556951646208', 'video_name': 'ElephantsDream', 'create_time': '1759039334664'}], 'current_page': 1, 'page_size': 200, 'total_count': '1'}, 'success': True, 'failed': False}


## Search

You can perform a **semantic search** across all uploaded videos to find the most relevant matches.

In this example, we’ll search for the video we just uploaded by providing a short text description of its content.


In [10]:
headers = {"Authorization": API_KEY} 
json_body = {
    "search_param": "Proog (an older guide) and Emo (a younger protege) explore a surreal, vast machine",  # The search query
    "search_type": "BY_VIDEO" # 'BY_AUDIO' or 'BY_VIDEO' or 'BY_CLIP'
}

response = requests.post(
    "https://api.memories.ai/serve/api/v1/search",
    headers=headers,
    json=json_body
)

print(response.json())

{'code': '0000', 'msg': 'success', 'data': [{'videoNo': 'VI627384556951646208', 'videoName': 'ElephantsDream', 'startTime': '467', 'endTime': '501', 'score': 0.6890185347401752}, {'videoNo': 'VI627384556951646208', 'videoName': 'ElephantsDream', 'startTime': '306', 'endTime': '340', 'score': 0.6814523928154906}, {'videoNo': 'VI627384556951646208', 'videoName': 'ElephantsDream', 'startTime': '424', 'endTime': '466', 'score': 0.6776493398948245}, {'videoNo': 'VI627384556951646208', 'videoName': 'ElephantsDream', 'startTime': '81', 'endTime': '101', 'score': 0.6756581009024786}, {'videoNo': 'VI627384556951646208', 'videoName': 'ElephantsDream', 'startTime': '190', 'endTime': '233', 'score': 0.67533732439739}, {'videoNo': 'VI627384556951646208', 'videoName': 'ElephantsDream', 'startTime': '293', 'endTime': '305', 'score': 0.6726333686576976}, {'videoNo': 'VI627384556951646208', 'videoName': 'ElephantsDream', 'startTime': '234', 'endTime': '247', 'score': 0.6683511406677032}, {'videoNo': 'V

## VideoChat Prompt Example

You can interact with the **Memories.AI Chat API** to ask questions about a specific video.

In the example below, we’ll send a prompt regarding highlights to the API and receive a response generated from the video’s parsed content.


In [ ]:
import json
headers = {
    "Authorization": API_KEY,
    "Content-Type": "application/json",
}

payload = {
    "video_nos": ["VI627384556951646208"],  # List of video IDs to chat about
    "prompt": "Find the highlights of this video and give timestamps"  # User query
}

response = requests.post(
    "https://api.memories.ai/serve/api/v1/chat",
    headers=headers,
    data=json.dumps(payload),
    stream=False
)

if response.status_code != 200:
    print(response.status_code)
    print(response.text)
else:
    try:
        for line in response.iter_lines(decode_unicode=True):
            if line:
                print(line)
                if line.strip().lower() == 'data:"done"':
                    print("\n")
                    break
                if line.startswith("data:"):
                    print(line.replace("data:", "").strip(), end="", flush=True)
    except Exception as e:
        print(str(e))

{"code":"0000","msg":"success","data":{"role":"ASSISTANT","content":"Here are the highlights with approximate timestamps and why each moment matters.\n\nApprox. runtime: ~10:34\nSource: \"Elephants Dream\" — Orange Open Movie Project\nQuoted line from the content: \"listen\" — Orange Open Movie Project\n\n1. 00:00–00:45 — Opening atmosphere and setup  \n    - Establishes the complex industrial world: wiring, metallic platforms and the uneasy tone.  \n    - Introduces Emo (the skeptical younger character) and the mood of disorientation.\n\n2. 00:45–02:15 — First encounter / Proog appears and the invitation to \"listen\"  \n    - Proog/Pruke engages Emo, repeatedly urging him to \"listen\" and pay attention to the perceived perfection around them.  \n    - Sets up the central theme: trust vs. skepticism.\n\n3. 02:15–04:00 — Journey through precarious platforms and mechanical creatures  \n    - Traversing suspended walkways and encountering robotic bird-like machines; visually rich, incre

# Video transcription
You can use the API to transcribe the video's visual content into text.

In [12]:
headers = {"Authorization": API_KEY}
params = {"video_no": "VI627384556951646208"}

response = requests.get("https://api.memories.ai/serve/api/v1/get_video_transcription", headers=headers, params=params)

print("Status:", response.status_code)
try:
    print("Video Transcription Response:", response.json())
except Exception:
    print("Response Text:", response.text)

Status: 200
Video Transcription Response: {'code': '0000', 'msg': 'success', 'data': {'videoNo': 'VI627384556951646208', 'transcriptions': [{'index': 0, 'content': 'The screen displays a textured wall with embossed text. The text reads "the orange open movie project presents". The wall appears to be made of cracked tiles and has various metallic and mechanical elements on its sides.', 'startTime': '0', 'endTime': '10'}, {'index': 1, 'content': 'The textured wall continues to display embossed text, now showing "elephants dream". The overall aesthetic remains consistent with cracked tiles and mechanical embellishments.', 'startTime': '10', 'endTime': '20'}, {'index': 2, 'content': "The scene shifts to a close-up of an animated character's face. The character is bald with prominent features and appears to be looking downwards with a concerned expression. The background is dark and blurred, with hints of orange and yellow light.", 'startTime': '26', 'endTime': '28'}, {'index': 3, 'content'

## Audio Transcription

Use the API to convert a video’s **spoken audio** into text.

> **Note:** The video must contain audible speech for transcription to work correctly.


In [13]:
headers = {"Authorization": API_KEY}
params = {"video_no": "VI627384556951646208"}

response = requests.get("https://api.memories.ai/serve/api/v1/get_audio_transcription", headers=headers, params=params)

print("Status:", response.status_code)
try:
    print("Video Transcription Response:", response.json())
except Exception:
    print("Response Text:", response.text)

Status: 200
Video Transcription Response: {'code': '0000', 'msg': 'success', 'data': {'videoNo': 'VI627384556951646208', 'transcriptions': [{'index': 0, 'content': ' Podcasting this kind of video is thank you, Kristen.', 'startTime': '0', 'endTime': '6'}, {'index': 1, 'content': ' Okay, slowly, Penguin, I bet they are back.', 'startTime': '6', 'endTime': '36'}, {'index': 2, 'content': ' Are you hurt?', 'startTime': '36', 'endTime': '48'}, {'index': 3, 'content': " I don't think so.", 'startTime': '48', 'endTime': '53'}, {'index': 4, 'content': ' You?', 'startTime': '54', 'endTime': '54'}, {'index': 5, 'content': " I'm okay.", 'startTime': '55', 'endTime': '56'}, {'index': 6, 'content': ' Get up.', 'startTime': '57', 'endTime': '58'}, {'index': 7, 'content': " Emo, it's not safe here.", 'startTime': '58', 'endTime': '61'}, {'index': 8, 'content': " Let's go.", 'startTime': '62', 'endTime': '63'}, {'index': 9, 'content': " What's next?", 'startTime': '64', 'endTime': '64'}, {'index': 10,

## Generate Video Summary

Use the API to create a concise summary of a video’s content.

The `type` parameter accepts:
- `"TOPIC"` 
- `"CHAPTER"` 


In [14]:
headers = {"Authorization": API_KEY}
params = {
    "video_no": "VI627384556951646208",
    "type": "TOPIC"
}

response = requests.get("https://api.memories.ai/serve/api/v1/generate_summary", headers=headers, params=params)

print("Status:", response.status_code)
try:
    print("Summary Response:", response.json())
except Exception:
    print("Response Text:", response.text)

Status: 200
Summary Response: {'code': '0000', 'msg': 'success', 'data': {'summary': "In a surreal and complex environment filled with intricate wires and metallic structures, two figures, an older man (Pruke) and a younger one (Emo), navigate a strange world while discussing its nature and dangers. Pruke tries to show Emo the beauty and perfection of their surroundings, but Emo expresses skepticism and a desire to leave, feeling that there is nothing of substance. They encounter various obstacles and unsettling elements, including mechanical creatures and ominous environments, leading to tension and disagreement between them. Pruke urges Emo to listen to the sounds of the machine and learn to trust their surroundings, while Emo questions Pruke's judgment and sanity. The journey involves themes of trust, perception, and the struggle to find meaning in a bizarre and potentially dangerous reality.", 'items': [{'description': '• Emo and Pruke are in a strange, complex environment filled w

# Utility
There are also utility APIs for managing and retrieving metadata about videos and sessions in the Memories.ai platform.

# List Videos
You can retrieve a paginated list of videos that have been uploaded to the platform. You can optionally filter the results by video name, video ID, folder, or processing status.

In [15]:
headers = {"Authorization": API_KEY}

json_body = {
    "page": 1,
    "size": 200,
    # "video_name": "<VIDEO_NAME>",
    # "video_no": "<VIDEO_ID>",
    # "unique": "<UNIQUE_ID>",
    # "status": "<STATUS>",
}

response = requests.post("https://api.memories.ai/serve/api/v1/list_videos", headers=headers, json=json_body)
print(response.json())

{'code': '0000', 'msg': 'success', 'data': {'videos': [{'duration': '653', 'size': '169612362', 'status': 'PARSE', 'cause': 'null', 'video_no': 'VI627384556951646208', 'video_name': 'ElephantsDream', 'create_time': '1759039334664'}, {'duration': '1487', 'size': '124332274', 'status': 'PARSE', 'cause': 'null', 'video_no': 'VI625685456915968000', 'video_name': 'Top 5 Most-Watched Live Sketches | Season 50 | Saturday Night Live', 'create_time': '1758634237659'}, {'duration': '47', 'size': '7849512', 'status': 'PARSE', 'cause': 'null', 'video_no': 'VI622833466192486400', 'video_name': 'paris-YT', 'create_time': '1757954270072'}, {'duration': '28', 'size': '5811353', 'status': 'PARSE', 'cause': 'null', 'video_no': 'VI622833446755811328', 'video_name': 'money-YT', 'create_time': '1757954265440'}, {'duration': '16', 'size': '2834756', 'status': 'PARSE', 'cause': 'null', 'video_no': 'VI622833431903780864', 'video_name': 'naruto-YT', 'create_time': '1757954261898'}, {'duration': '12', 'size': '

# List Chat Sessions
You can retrieve a paginated list of historical chat sessions, including metadata such as session IDs, timestamps, and related videos.

In [30]:
headers = {"Authorization": API_KEY}

params = {
    "page": "1",
    "unique_id": "default"
}

response = requests.get("https://api.memories.ai/serve/api/v1/list_sessions", headers=headers, params=params)
print(response.json())

{'code': '0000', 'msg': 'success', 'data': {'sessions': [{'sessionId': '627385279038828544', 'title': 'Find the highlights of this video and give timestamps'}, {'sessionId': '611587302624489472', 'title': 'Find the highlights of this video and give timestamps'}, {'sessionId': '611482004551131138', 'title': 'where is the war?'}, {'sessionId': '611479007301763072', 'title': 'Find the highlights of this video and give timestamps'}, {'sessionId': '611477893944737792', 'title': 'Find the highlights of this video and give timestamps'}, {'sessionId': '611476482775216128', 'title': 'Generate summaries of these videos'}, {'sessionId': '611476130202292224', 'title': 'Generate hashtags and titles for this video'}], 'current_page': 1, 'page_size': 20, 'total_count': '7'}, 'success': True, 'failed': False}


# Get Session Detail
You can retrieve detailed metadata about a specific video session.
From last block, we can choose a session Id.

In [34]:
headers = {"Authorization": API_KEY}
params = {"session_id": "611587302624489472", "unique_id": "default" }
url = "https://api.memories.ai/serve/api/v1/get_session_detail"

response = requests.get(url, headers=headers, params=params)

if response.status_code == 200:
    print("Session detail retrieved successfully:")
    print(response.json())
else:
    print("Failed to retrieve session detail:", response.status_code)
    print(response.text)

Session detail retrieved successfully:
{'code': '0000', 'msg': 'success', 'data': {'title': 'Find the highlights of this video and give timestamps', 'messages': [{'role': 'user', 'content': 'Find the highlights of this video and give timestamps'}, {'role': 'assistant', 'content': 'Here are the key highlights with timestamps (approximate) and why each moment matters — prioritized by narrative importance.\n\nHighlights & timestamps\n- 00:00–00:30 — Opening atmosphere\n  - What happens: Opens on a surreal, metallic world of wires, lights and suspended platforms that establishes tone and scale.\n  - Why it matters: Sets the film’s mood and visual language — ominous, mechanical, labyrinthine.\n\n- 00:30–02:00 — Introduction of the two characters\n  - What happens: Proog (older, bald man in a dark green coat) and Emo (younger, gaunt man in a light tunic) appear and begin interacting.\n  - Why it matters: Establishes the mentor/guide dynamic and their contrasting attitudes toward the machine.

# Delete Video
You can delete all raw and derived data associated with the specified videoNos in the request. Once the API call is successfully completed, no data related to the deleted videos will be retained.

In [ ]:
headers = {"Authorization": API_KEY}  
# List of video IDs to delete
data = ["VI611165958140055552","VI611154695637032960"]

response = requests.post(
    "https://api.memories.ai/serve/api/v1/delete_videos",
    headers=headers,
    json=data
)

print(response.json())

{'code': '0000', 'msg': 'success', 'data': None, 'success': True, 'failed': False}
